# Construction du Pipeline ML MLlib

## Chargement des données

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("gestionlogistique").getOrCreate()
df = spark.read.option("header", "true").parquet("data/output_training_parquet/part-00000-97d86afc-357a-4d4f-96af-144a5770788b-c000.snappy.parquet")

df.show()
df.printSchema()

+-----------------+------------------+------------------+-------------------+------------------------+-------------------+------------------+----------------------+------------------+-----------------+-------------+------------------+----------------+-----------------+-------------------+-----------------+--------------------+-----------------------+
|Benefit per order|Sales per customer|Late_delivery_risk|Order Item Quantity|Order Item Product Price|Order Item Discount|  Order Item Total|Order Profit Per Order|          distance|Order Country_ohe|     Type_ohe|Customer State_ohe|Order Region_ohe|Shipping Mode_ohe|Department Name_ohe|Category Name_ohe|    numeric_features|scaled_numeric_features|
+-----------------+------------------+------------------+-------------------+------------------------+-------------------+------------------+----------------------+------------------+-----------------+-------------+------------------+----------------+-----------------+-------------------+-----

25/11/20 10:13:22 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## Conversion des colonnes numériques

In [3]:
from pyspark.ml.feature import VectorAssembler

target_column = "Late_delivery_risk"

feature_cols = [
    "Benefit per order",
    "Sales per customer",
    "Order Item Quantity",
    "Order Item Product Price",
    "Order Item Discount",
    "Order Item Total",
    "Order Profit Per Order",
    "distance",
    "Order Country_ohe",
    "Type_ohe",
    "Customer State_ohe",
    "Order Region_ohe",
    "Shipping Mode_ohe",
    "Department Name_ohe",
    "Category Name_ohe",
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_vec = assembler.transform(df).select("features", target_column)


## Random Forest AMÉLIORÉ avec hyperparamètres optimisés

In [4]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

train_v2, test_v2 = df_vec.randomSplit([0.8, 0.2], seed=42)

rf_improved = RandomForestClassifier(
    labelCol=target_column,
    featuresCol="features",
    numTrees=200,
    maxDepth=10,
    minInstancesPerNode=10,
    maxBins=64,
    subsamplingRate=0.8,
    seed=42
)

model_improved = rf_improved.fit(train_v2)
pred_improved = model_improved.transform(test_v2)

roc_improved = BinaryClassificationEvaluator(labelCol="Late_delivery_risk", metricName="areaUnderROC").evaluate(pred_improved)
accuracy_improved = MulticlassClassificationEvaluator(labelCol="Late_delivery_risk", metricName="accuracy").evaluate(pred_improved)
f1_improved = MulticlassClassificationEvaluator(labelCol="Late_delivery_risk", metricName="f1").evaluate(pred_improved)

print(f"AUC ROC = {roc_improved:.4f}")
print(f"Accuracy = {accuracy_improved:.4f}")
print(f"F1 Score = {f1_improved:.4f}")

25/11/19 22:30:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/11/19 22:31:44 WARN DAGScheduler: Broadcasting large task binary with size 1442.8 KiB
25/11/19 22:31:44 WARN DAGScheduler: Broadcasting large task binary with size 1442.8 KiB
25/11/19 22:31:59 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
25/11/19 22:31:59 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
25/11/19 22:32:16 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB
25/11/19 22:32:16 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB
25/11/19 22:32:35 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB
25/11/19 22:32:35 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB
25/11/19 22:32:56 WARN DAGScheduler: Broadcasting large task binary with size 5.2 MiB
25/11/19 22:32:56 WARN DAGSched

AUC ROC = 0.7500
Accuracy = 0.6934
F1 Score = 0.6913


## Gradient Boosting Classifier (GBT)


In [5]:
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

train_gbt, test_gbt = df_vec.randomSplit([0.8, 0.2], seed=42)

gbt = GBTClassifier(
    labelCol=target_column,
    featuresCol="features",
    maxIter=150,
    maxDepth=8,
    stepSize=0.05,
    subsamplingRate=0.8,
    seed=42
)

gbt_model = gbt.fit(train_gbt)
gbt_pred = gbt_model.transform(test_gbt)

gbt_roc = BinaryClassificationEvaluator(labelCol="Late_delivery_risk", metricName="areaUnderROC").evaluate(gbt_pred)
gbt_accuracy = MulticlassClassificationEvaluator(labelCol="Late_delivery_risk", metricName="accuracy").evaluate(gbt_pred)
gbt_f1 = MulticlassClassificationEvaluator(labelCol="Late_delivery_risk", metricName="f1").evaluate(gbt_pred)

print(f"AUC ROC = {gbt_roc:.4f}")
print(f"Accuracy = {gbt_accuracy:.4f}")
print(f"F1 Score = {gbt_f1:.4f}")

25/11/19 22:38:09 WARN DAGScheduler: Broadcasting large task binary with size 1003.4 KiB
25/11/19 22:38:09 WARN DAGScheduler: Broadcasting large task binary with size 1003.4 KiB
25/11/19 22:38:09 WARN DAGScheduler: Broadcasting large task binary with size 1012.8 KiB
25/11/19 22:38:09 WARN DAGScheduler: Broadcasting large task binary with size 1012.8 KiB
25/11/19 22:38:09 WARN DAGScheduler: Broadcasting large task binary with size 1012.9 KiB
25/11/19 22:38:09 WARN DAGScheduler: Broadcasting large task binary with size 1012.9 KiB
25/11/19 22:38:10 WARN DAGScheduler: Broadcasting large task binary with size 1013.4 KiB
25/11/19 22:38:10 WARN DAGScheduler: Broadcasting large task binary with size 1013.4 KiB
25/11/19 22:38:11 WARN DAGScheduler: Broadcasting large task binary with size 1014.2 KiB
25/11/19 22:38:11 WARN DAGScheduler: Broadcasting large task binary with size 1014.2 KiB
25/11/19 22:38:11 WARN DAGScheduler: Broadcasting large task binary with size 1015.3 KiB
25/11/19 22:38:11 WAR

AUC ROC = 0.7688
Accuracy = 0.7051
F1 Score = 0.7009


## Logistic Regression (Baseline)


In [13]:
from pyspark.ml.classification import LogisticRegression
train_gbt, test_gbt = df_vec.randomSplit([0.8, 0.2], seed=42)

lr = LogisticRegression(
    labelCol=target_column,
    featuresCol="features",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.5
)

print("📈 Entraînement du modèle Logistic Regression...")
lr_model = lr.fit(train_gbt)
lr_pred = lr_model.transform(test_gbt)

lr_roc = BinaryClassificationEvaluator(
    labelCol="Late_delivery_risk",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(lr_pred)

lr_accuracy = MulticlassClassificationEvaluator(
    labelCol="Late_delivery_risk",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(lr_pred)

lr_f1 = MulticlassClassificationEvaluator(
    labelCol="Late_delivery_risk",
    predictionCol="prediction",
    metricName="f1"
).evaluate(lr_pred)

print(f"\n📊 Résultats Logistic Regression:")
print(f"AUC ROC = {lr_roc:.4f}")
print(f"Accuracy = {lr_accuracy:.4f}")
print(f"F1 Score = {lr_f1:.4f}")

📈 Entraînement du modèle Logistic Regression...



📊 Résultats Logistic Regression:
AUC ROC = 0.7428
Accuracy = 0.6963
F1 Score = 0.6932


## Sauvegarde sécurisée du modèle 


In [ ]:
path = "models/gbt_model"
gbt_model.write().overwrite().save(path)
